In [16]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from xgboost import XGBClassifier

PROJECT_ROOT = Path(r"D:\Python project\PROJECT")
MODELS_DIR = PROJECT_ROOT / "models"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [17]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

In [18]:
ae_checkpoint = torch.load(
    MODELS_DIR / "autoencoder_full.pt",
    map_location=device,
    weights_only=False
)

ae_model = Autoencoder(
    input_dim=ae_checkpoint["input_dim"]
).to(device)

ae_model.load_state_dict(
    ae_checkpoint["model_state_dict"]
)

ae_model.eval()

scaler = joblib.load(
    MODELS_DIR / "autoencoder_scaler.joblib"
)

with open(
    MODELS_DIR / "anomaly_threshold.json",
    "r"
) as f:
    ae_metadata = json.load(f)

ae_threshold = ae_metadata["threshold"]
ae_feature_cols = ae_metadata["feature_columns"]

print("Autoencoder loaded")
print("Input dimension:", ae_checkpoint["input_dim"])
print("Threshold:", ae_threshold)

Autoencoder loaded
Input dimension: 52
Threshold: 0.507518


In [19]:
xgb_model = XGBClassifier()

xgb_model.load_model(
    MODELS_DIR / "xgboost_classifier.json"
)

label_encoder = joblib.load(
    MODELS_DIR / "xgboost_label_encoder.joblib"
)

with open(
    MODELS_DIR / "xgboost_metadata.json",
    "r"
) as f:
    xgb_metadata = json.load(f)

xgb_feature_cols = xgb_metadata["feature_columns"]

print("XGBoost loaded")
print("Number of classes:", len(label_encoder.classes_))
print("Classes:", label_encoder.classes_)

XGBoost loaded
Number of classes: 7
Classes: ['Bots' 'Brute Force' 'DDoS' 'DoS' 'Normal Traffic' 'Port Scanning'
 'Web Attacks']


In [20]:
assert ae_feature_cols == xgb_feature_cols

feature_cols = xgb_feature_cols

print("Feature order: OK")
print("Number of features:", len(feature_cols))

Feature order: OK
Number of features: 52


In [21]:
test_df = pd.read_csv(
    PROJECT_ROOT / "data" / "splits" / "test.csv",
    low_memory=False
)

sample_idx = test_df.index[
    test_df["Attack Type"] != "Normal Traffic"
][0]

one_flow = test_df.loc[
    [sample_idx],
    feature_cols
]

true_label = test_df.loc[
    sample_idx,
    "Attack Type"
]

print("Flow shape:", one_flow.shape)
print("True label:", true_label)
print("Selected index:", sample_idx)
print("True label:", test_df.loc[sample_idx, "Attack Type"])

Flow shape: (1, 52)
True label: DoS
Selected index: 12
True label: DoS


In [22]:
one_flow_scaled = scaler.transform(
    one_flow
).astype("float32")

one_flow_tensor = torch.tensor(
    one_flow_scaled,
    dtype=torch.float32
).to(device)

with torch.no_grad():
    reconstructed = ae_model(one_flow_tensor)

anomaly_score = torch.mean(
    (one_flow_tensor - reconstructed) ** 2,
    dim=1
).item()

is_anomaly = anomaly_score > ae_threshold

print("Anomaly score:", anomaly_score)
print("Threshold:", ae_threshold)
print("Is anomaly:", is_anomaly)

Anomaly score: 2.819601535797119
Threshold: 0.507518
Is anomaly: True


In [23]:
predicted_id = (
    xgb_model
    .predict(one_flow)
    .astype(int)
    .ravel()[0]
)

probabilities = (
    xgb_model
    .predict_proba(one_flow)
    .ravel()
)

predicted_label = label_encoder.inverse_transform(
    [predicted_id]
)[0]

confidence = probabilities[predicted_id]

print("Predicted label:", predicted_label)
print("Confidence:", confidence)

Predicted label: DoS
Confidence: 0.9999188


In [24]:
probability_df = pd.DataFrame({
    "Attack Type": label_encoder.classes_,
    "Probability": probabilities
}).sort_values(
    "Probability",
    ascending=False
)

result = {
    "true_label": true_label,
    "anomaly_score": anomaly_score,
    "is_anomaly": is_anomaly,
    "predicted_attack": predicted_label,
    "confidence": float(confidence)
}

print(result)
display(probability_df)

{'true_label': 'DoS', 'anomaly_score': 2.819601535797119, 'is_anomaly': True, 'predicted_attack': 'DoS', 'confidence': 0.9999188184738159}


,Attack Type,Probability
3,DoS,9.999188e-01
4,Normal Traffic,6.305110e-05
2,DDoS,9.248878e-06
5,Port Scanning,6.737576e-06
1,Brute Force,9.999749e-07
0,Bots,7.798544e-07
6,Web Attacks,3.789531e-07


In [25]:

sample_idx = test_df.index[
    test_df["Attack Type"] != "Normal Traffic"
][0]

print("Selected index:", sample_idx)
print("True label:", test_df.loc[sample_idx, "Attack Type"])

Selected index: 12
True label: DoS


In [28]:
def analyze_flow(flow_df, true_label=None):
    # Chỉ lấy 52 feature, bỏ nhãn thật
    x_raw = flow_df[feature_cols]

    # Autoencoder cần dữ liệu đã scale
    x_scaled = scaler.transform(
        x_raw
    ).astype("float32")

    x_tensor = torch.tensor(
        x_scaled,
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        reconstructed = ae_model(x_tensor)

    anomaly_score = torch.mean(
        (x_tensor - reconstructed) ** 2,
        dim=1
    ).item()

    is_anomaly = anomaly_score > ae_threshold

    # XGBoost dùng dữ liệu raw
    predicted_id = int(
        xgb_model.predict(x_raw)[0]
    )

    probabilities = (
        xgb_model
        .predict_proba(x_raw)[0]
    )

    predicted_label = label_encoder.inverse_transform(
        [predicted_id]
    )[0]

    result = {
        "true_label": true_label,
        "anomaly_score": anomaly_score,
        "is_anomaly": is_anomaly,
        "predicted_attack": predicted_label,
        "confidence": float(
            probabilities[predicted_id]
        )
    }
    result["attack_probabilities"] = {
    label: float(probability)
    for label, probability in zip(
        label_encoder.classes_,
        probabilities
    )
}

    return result

In [29]:
sample_rows = (
    test_df
    .groupby("Attack Type", sort=False)
    .head(1)
)

results = []

for idx, row in sample_rows.iterrows():
    result = analyze_flow(
        test_df.loc[[idx]],
        true_label=row["Attack Type"]
    )

    results.append(result)

smoke_test_df = pd.DataFrame(results)

display(smoke_test_df)

,true_label,anomaly_score,is_anomaly,predicted_attack,confidence,attack_probabilities
0,Normal Traffic,0.000971,False,Normal Traffic,0.999986,"{'Bots': 2.1520486370718572e-07, 'Brute Force'..."
1,DoS,2.819602,True,DoS,0.999919,"{'Bots': 7.798544174875133e-07, 'Brute Force':..."
2,Port Scanning,0.026459,False,Port Scanning,0.944720,"{'Bots': 3.0575927667086944e-05, 'Brute Force'..."
3,DDoS,0.007793,False,DDoS,0.999485,"{'Bots': 2.745553899785591e-07, 'Brute Force':..."
4,Brute Force,0.003938,False,Brute Force,0.999894,"{'Bots': 2.647601471394978e-09, 'Brute Force':..."
5,Bots,0.016895,False,Bots,1.000000,"{'Bots': 0.9999997615814209, 'Brute Force': 3...."
6,Web Attacks,0.005569,False,Web Attacks,0.935759,"{'Bots': 1.8399816781311529e-06, 'Brute Force'..."
